# Divine-name networks in the Baal Cycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/master/notebooks/3c_divine_name_networks.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/master?labpath=notebooks%2F3c_divine_name_networks.ipynb)

*Hour 3 · alternative network lab*

> A **graph** consists of nodes and connections. A **clique** is a group of
> nodes in which every node is connected to every other node.

## Historical question

Which divine figures are narratively brought into one another’s orbit in
**KTU 1.1–1.6**, the tablets conventionally associated with the Baal Cycle?

We will turn proximity in the text into a graph, search that graph for tightly
connected groups, and then return to the passages that created those
connections.


# The model in one sentence

> Two divine names are connected when they occur in the **same tablet and
> column**, no more than **three numbered lines apart**.

| Graph element | Philological interpretation |
|---|---|
| node | a normalized name tagged `DN` |
| edge | at least one occurrence within three lines |
| edge thickness | number of supporting line-pair windows |
| node size | number of mentions |
| clique | every pair of names is connected somewhere in the epic |

This is a graph of **textual proximity**. An edge does not by itself mean
friendship, kinship, cultic association, or even direct interaction.


## What the notebook code will show

The visible code follows the research argument:

1. load the divine names;
2. build a graph using the three-line rule;
3. inspect the passages behind an edge;
4. draw the graph;
5. find and audit cliques;
6. test a different distance rule.

Parsing TSV rows, reconciling transliterations, storing provenance, and
generating the interactive HTML are implementation details. They live in
[`workshop_tools/divine_networks.py`](../workshop_tools/divine_networks.py),
where they remain inspectable without interrupting the lab narrative.


## 0. Setup

In [1]:
# === SETUP — run once ===
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pandas", "networkx", "plotly"],
        check=False,
    )

sys.path.insert(0, os.path.abspath(".."))

from IPython.display import display
from workshop_tools.divine_networks import (
    build_divine_name_graph,
    clique_evidence,
    edge_evidence,
    edge_passages,
    filter_divine_name_graph,
    find_maximal_cliques,
    graph_overview,
    graph_sensitivity,
    load_baal_cycle,
    DEFAULT_ALIASES,
    local_name_windows,
    plot_graph_sensitivity,
    show_divine_name_network,
)


## 1. Load the annotated names

`load_baal_cycle()` reads KTU 1.1–1.6 and keeps tokens tagged `DN`. It also:

- joins common allonyms such as Haddu/Baal, River/Sea, and Kothar/Ḫasis;
- attaches descriptions from the onomastic glossary;
- retains the original surface forms and glosses for audit.

Those mechanics are hidden here, but the resulting editorial decisions are not.


In [2]:
corpus = load_baal_cycle()

display(corpus.overview().to_frame("value"))
display(corpus.name_summary.head(20))


,value
tablets,6
DN occurrences,612
normalized names,37
names with onomastic descriptions,28


,mentions,description,description_source
deity,,,
bˤl,156,Baʿlu/Baal,onomastic override
il,98,ʾIlu/Ilu/El,onomastic override
ym,66,Yammu/the god “Sea”,onomastic override
kṯr,46,Koṯaru/Kothar,onomastic override
aṯrt,44,ʾAṯiratu/Athirat/Asherah/El’s wife,onomastic override
ˤnt,37,ʿAnatu/Anat,onomastic override
mt,36,Mot/Death,onomastic override
ṣpn,17,Ṣapānu/Zaphon/the mountain dwelling of bʕl dei...,onomastic override
špš,16,Šapšu/Shapsh/Shapshu,onomastic override


### Entity resolution is interpretation

A computer initially sees `hd`, `bˤl`, `nhr`, `ym`, `kṯr`, and `ḫss`
as six unrelated strings. The table below records every automatic merger.
Review it before interpreting the network.

Try removing an alias later: the graph should change when the philological
assumption changes.


In [3]:
display(corpus.normalization_changes)


,surface,deity,gloss
0,aṯ,aṯrt,Asherah
1,aṯtrt,aṯrt,Asherah
2,rt,aṯrt,Asherah
3,bˤlm,bˤl,Baʿlu/Baal
4,dmrn,bˤl,the Powerful One/Valiant One/title of Baal
5,hd,bˤl,Haddu/Hadad
6,hdxt,bˤl,Haddu/Hadad
7,lbˤl,bˤl,"to, towards, up to; lord; owner of (or residen..."
8,ˤlm,bˤl,Baʿlu/Baal
9,id,il,ʾIlu/Ilu/El; the one(s) of


## 2. Build the graph

The distance is the only analytical rule needed here. Names are compared inside
one tablet and one column, so the end of a column is never treated as adjacent
to the beginning of the next.


In [4]:
G = build_divine_name_graph(corpus, max_distance=3)

display(graph_overview(G).to_frame("value"))


,value
nodes,37.000
edges,146.000
density,0.219
line distance,3.000


## 3. Read an edge back into the text

A network is useful only if its claims remain traceable. Let us inspect the
connection between Baal (`bˤl`) and Sea (`ym`).

The first table lists supporting line-pair windows. The second reconstructs one
of those passages with a line of context.


In [5]:
display(edge_evidence(G, "bˤl", "ym").head(12))
display(edge_passages(corpus, G, "bˤl", "ym", limit=1, padding=1))


,reference,tablet,column,first,last
0,KTU 1.2 I:4–7,1.2,I,4,7
1,KTU 1.2 I:7–8,1.2,I,7,8
2,KTU 1.2 I:17–18,1.2,I,17,18
3,KTU 1.2 I:21–22,1.2,I,21,22
4,KTU 1.2 I:22–24,1.2,I,22,24
5,KTU 1.2 I:24–26,1.2,I,24,26
6,KTU 1.2 I:33–35,1.2,I,33,35
7,KTU 1.2 I:33–36,1.2,I,33,36
8,KTU 1.2 I:34–35,1.2,I,34,35
9,KTU 1.2 I:34–36,1.2,I,34,36


,edge_window,reference,text
0,KTU 1.2 I:4–7,KTU 1.2 I:3,at ypˤt b a
1,KTU 1.2 I:4–7,KTU 1.2 I:4,aliyn bˤl
2,KTU 1.2 I:4–7,KTU 1.2 I:5,drk tk tk ṯmšl
3,KTU 1.2 I:4–7,KTU 1.2 I:6,b rišk aymr
4,KTU 1.2 I:4–7,KTU 1.2 I:7,ṯpṭ nhr yṯbr
5,KTU 1.2 I:4–7,KTU 1.2 I:8,rišk ˤṯtrt šm bˤl qdqd


## 4. Draw the network

For readability, the picture shows names attested at least twice and edges
supported by at least three windows. This visual filter does **not** alter the
graph used for clique analysis below.

- Drag nodes to rearrange them.
- Scroll to zoom.
- Hover over a node for its description.
- Hover over an edge for its weight and sample KTU references.
- Use **Freeze layout**, **Resume physics**, and **Fit** above the graph.

Node size represents mentions; edge thickness represents supporting windows.
Edge length is merely a consequence of the layout and has no independent
meaning.


In [6]:
show_divine_name_network(G, min_mentions=2, min_weight=3)


## 5. Find maximal cliques

A clique is a set in which every pair of names is connected. A clique is
**maximal** when no further node can be added without breaking that condition.
“Maximal” does not necessarily mean “largest.”

We exclude names attested only once, because a single damaged or uncertain form
should not generate a headline result.


In [7]:
cliques = find_maximal_cliques(G, min_mentions=2, min_size=3)

display(cliques.drop(columns="members").head(20))


,size,pair_weight_sum,names
0,8,247,"arṣy, aṯrt, bˤl, il, pdry, yˤbdr, ˤnt, ṭly"
1,7,318,"aṯrt, bˤl, dgn, il, qdš, ym, ˤnt"
2,7,305,"aṯrt, bˤl, il, pdry, ym, ˤnt, ṭly"
3,7,289,"aṯrt, bˤl, dgn, il, qdš, ym, ˤṯtr"
4,6,276,"aṯrt, bˤl, il, kṯr, ym, ṭly"
5,6,255,"bˤl, dgn, il, ym, špš, ˤnt"
6,6,227,"bˤl, dgn, il, ym, špš, ˤṯtr"
7,6,226,"aṯrt, bˤl, gpn, il, ugr, ym"
8,6,216,"bˤl, il, mt, špš, ˤnt, ṣpn"
9,6,189,"amrr, aṯrt, bˤl, il, qdš, ˤnt"


### Audit the largest clique

There is an important trap. A clique in the aggregated graph does **not** imply
that all its names occur together in one scene. Each pair may have acquired its
edge in a different passage—or even a different tablet.

Expand the largest clique into its pairwise claims. Weak edges at the top of
the table reveal which sparse connections hold the large structure together.


In [8]:
largest = cliques.iloc[0]["members"]

display(clique_evidence(G, largest))


,pair,weight,example_windows
0,arṣy — bˤl,1,KTU 1.3 III:6–7
1,bˤl — yˤbdr,1,KTU 1.3 III:6–8
2,arṣy — ˤnt,2,KTU 1.3 III:7–9; KTU 1.3 IV:51–53
3,il — yˤbdr,2,KTU 1.3 IV:52–54; KTU 1.4 IV:57–58
4,pdry — ˤnt,2,KTU 1.3 III:6–9; KTU 1.3 IV:50–53
5,ˤnt — ṭly,2,KTU 1.3 III:7–9; KTU 1.3 IV:51–53
6,arṣy — il,3,KTU 1.3 IV:48–51; KTU 1.3 IV:51–54; KTU 1.4 IV...
7,bˤl — ṭly,3,KTU 1.3 I:21–24; KTU 1.3 I:22–24; KTU 1.3 III:6–7
8,yˤbdr — ˤnt,3,KTU 1.3 III:8–9; KTU 1.3 III:8–11; KTU 1.3 IV:...
9,aṯrt — yˤbdr,4,KTU 1.3 IV:49–52; KTU 1.3 V:40–43; KTU 1.4 I:1...


## 6. Which groups really occur in one four-line window?

Now ask the stronger question readers often assume a clique already answers:
which sets of names are all contained between line *n* and line *n+3* in one
column?

Compare the largest local group with the largest aggregated clique.


In [9]:
local_groups = local_name_windows(
    corpus, max_distance=3, min_mentions=2, min_size=3
)

display(local_groups.drop(columns="members").head(20))
print(
    "largest aggregated clique:", cliques.iloc[0]["size"],
    "· largest single-window group:", local_groups.iloc[0]["size"],
)


,size,reference,names
0,6,KTU 1.2 I:37–40,"bˤl, dgn, qdš, ym, ˤnt, ˤṯtrt"
1,6,KTU 1.3 III:6–9,"arṣy, bˤl, pdry, yˤbdr, ˤnt, ṭly"
2,5,KTU 1.2 I:19–22,"bˤl, dgn, il, qdš, ym"
3,5,KTU 1.2 I:35–38,"bˤl, dgn, il, qdš, ym"
4,5,KTU 1.2 I:36–39,"bˤl, dgn, il, qdš, ym"
5,5,KTU 1.2 I:38–41,"bˤl, qdš, ym, ˤnt, ˤṯtrt"
6,5,KTU 1.3 III:36–39,"bˤl, gpn, il, ugr, ym"
7,5,KTU 1.3 IV:48–51,"arṣy, aṯrt, il, pdry, ṭly"
8,5,KTU 1.3 IV:49–52,"arṣy, aṯrt, pdry, yˤbdr, ṭly"
9,5,KTU 1.3 IV:50–53,"arṣy, pdry, yˤbdr, ˤnt, ṭly"


largest aggregated clique: 8 · largest single-window group: 6


## 7. Why three lines?

Three lines is a modelling choice, not a property of the tablets. Rebuild the
graph using distances from zero (same line only) through five. A conclusion
that appears only at one threshold should be reported as threshold-dependent.


In [10]:
sensitivity = graph_sensitivity(corpus, distances=range(0, 6))

display(sensitivity.round({"density": 3}))
plot_graph_sensitivity(sensitivity).show()


,distance,nodes,edges,density,largest_clique
0,0,27,31,0.088,4
1,1,27,78,0.222,5
2,2,27,107,0.305,7
3,3,27,121,0.345,8
4,4,27,135,0.385,8
5,5,27,140,0.399,8


## 8. Exercise

Work in pairs and keep a record of every interpretive decision.

1. **Read the evidence.** Choose one of the three largest cliques. Inspect two
   weak and two strong edges with `edge_evidence()` and
   `edge_passages()`. What kinds of passages create them?
2. **Compare meanings.** Contrast that aggregated clique with one entry in
   `local_groups`. Which is better described as a “scene,” and why?
3. **Challenge entity resolution.** Copy `DEFAULT_ALIASES`, remove one
   merger—for example `nhr → ym` or `ḫss → kṯr`—and pass the modified
   dictionary to `load_baal_cycle(aliases=...)`. Rebuild the graph. Which
   clique changes?
4. **Challenge the threshold.** Compare distances 1, 3, and 5. Name one stable
   conclusion and one threshold-dependent conclusion.
5. **Write a source-critical claim** in 3–4 sentences. It must name the
   computational rule, cite at least one KTU passage, and state one limitation.

### Optional stricter model

Use `filter_divine_name_graph(G, min_weight=2)` to require two evidential
windows, or `min_tablets=2` to require two tablets, before searching for
cliques. Does the result look more like a stable cast of characters and less
like a single crowded scene?


## 9. Discussion

- What is gained by using the annotation tag `DN` rather than a hand-built
  list?
- Which categories blur “deity,” “place,” “monster,” “weapon,” and “epithet”?
- Does a large clique reveal narrative structure, or mostly the frequency of
  Baal, El, and their allonyms?
- How do lacunae remove possible edges?
- Why is preserving edge provenance more important here than simply reporting
  a centrality score?

**Take-away:** the graph is an index back into the text. Clique detection
proposes passages and constellations to read; philological interpretation
decides what those constellations mean.
